In [0]:
%pip install openpyxl

In [0]:
# ==============================================================================
# BLOCO 1: IMPORTAÇÃO DE BIBLIOTECAS GLOBAIS
# ==============================================================================

import pandas as pd
import numpy as np
import re
import random
import os

print("Bibliotecas importadas com sucesso.")

In [0]:
# ==============================================================================
# BLOCO 2 CARREGAMENTO DA BASE DE DADOS (ARQUIVO EXCEL)
# ==============================================================================

# Caminho do diretório (volume)
diretorio = "/Volumes/workspace/default/mestrado_valiacao_requisitos/"

# Lista os ficheiros no diretório e encontra automaticamente o Excel
arquivos_no_volume = os.listdir(diretorio)
ficheiro_excel = [f for f in arquivos_no_volume if f.endswith('.xlsx')][0]

# Constrói o caminho exato
caminho_arquivo = os.path.join(diretorio, ficheiro_excel)
print(f"Ficheiro localizado no sistema: {caminho_arquivo}")

# Leitura do arquivo Excel
df_ahp = pd.read_excel(caminho_arquivo)

print(f"Base de dados carregada com sucesso. Total de avaliações lidas: {len(df_ahp)}")

# Exibe as primeiras linhas para validação visual
display(df_ahp.head())

PROCESSAMENTO AHP E GERAÇÃO DAS TABELAS FINAIS Os julgamentos individuais foram consolidados por meio da média geométrica das comparações par a par (Aggregation of Individual Judgments — AIJ), preservando a propriedade de reciprocidade das matrizes, e os vetores de prioridade relativa e as Razões de Consistência (CR) foram obtidos mediante normalização das colunas e aplicação dos Índices Aleatórios de Saaty. 

In [0]:
# ==============================================================================
# BLOCOS 3 E 4 PROCESSAMENTO AHP E GERAÇÃO DAS TABELAS FINAIS
# ==============================================================================


# 1. Função de conversão da escala
def converter_escala_saaty(texto):
    if pd.isna(texto): 
        return 1.0
    texto_str = str(texto)
    busca = re.search(r'\((\d+)\)', texto_str)
    if not busca: 
        return 1.0
    valor = float(busca.group(1))
    if 'Elem. B' in texto_str: 
        return 1.0 / valor
    return valor

# 2. Separação dos blocos de análise
colunas_comparacao = [col for col in df_ahp.columns if ' x ' in col]
colunas_rf = [c for c in colunas_comparacao if 'RF' in c and 'RNF' not in c]
colunas_rnf = [c for c in colunas_comparacao if 'RNF' in c]

# 3. Função central de cálculo e formatação
def calcular_ahp_por_bloco(colunas_bloco):
    criterios = set()
    pares = []
    for col in colunas_bloco:
        match = re.search(r'\[(.*?)\] x \[(.*?)\]', col)
        if match:
            crit_a, crit_b = match.group(1), match.group(2)
            criterios.update([crit_a, crit_b])
            pares.append((col, crit_a, crit_b))

    criterios = sorted(list(criterios))
    n = len(criterios)
    matriz_ahp = np.ones((n, n))

    for col, crit_a, crit_b in pares:
        valores_numericos = df_ahp[col].apply(converter_escala_saaty).tolist()
        media_geom = np.prod(valores_numericos) ** (1.0 / len(valores_numericos))
        i, j = criterios.index(crit_a), criterios.index(crit_b)
        matriz_ahp[i, j] = media_geom
        matriz_ahp[j, i] = 1.0 / media_geom

    soma_colunas = matriz_ahp.sum(axis=0)
    matriz_normalizada = matriz_ahp / soma_colunas
    pesos = matriz_normalizada.mean(axis=1)

    lambda_max = np.sum(soma_colunas * pesos)
    ci = (lambda_max - n) / (n - 1)
    ri_dict = {1: 0.0, 2: 0.0, 3: 0.58, 4: 0.90, 5: 1.12, 6: 1.24, 7: 1.32, 8: 1.41, 9: 1.45, 10: 1.49}
    ri = ri_dict.get(n, 1.49)
    cr = ci / ri if ri > 0 else 0

    # Estruturação visual idêntica à dissertação
    df_pesos = pd.DataFrame({"ID_Requisito": criterios, "Peso Relativo": pesos})
    df_pesos = df_pesos.sort_values(by="Peso Relativo", ascending=False).reset_index(drop=True)
    
    df_pesos.index = [f"{i+1}º" for i in range(len(df_pesos))]
    df_pesos.index.name = "Posição"
    
    df_pesos[['ID', 'Requisito']] = df_pesos['ID_Requisito'].str.split(' - ', n=1, expand=True)
    df_pesos['Peso Relativo'] = df_pesos['Peso Relativo'].apply(lambda x: f"{x:.2%}")
    
    return df_pesos[['ID', 'Requisito', 'Peso Relativo']], cr

# 4. Processamento dos dados
df_rnf, cr_rnf = calcular_ahp_por_bloco(colunas_rnf)
df_rf, cr_rf = calcular_ahp_por_bloco(colunas_rf)

# 5. Exibição das Tabelas Finais
print("TABELA RESULTADO DO PROCESSAMENTO AHP PARA 5 REQUISITOS NÃO FUNCIONAIS (RNF01 a RNF05)")
print(f"Razão de Consistência Matemática apurada em {cr_rnf:.3f}\n")
display(df_rnf)

print("\n" + "="*80 + "\n")

print("TABELA RESULTADO DO PROCESSAMENTO AHP PARA 7 REQUISITOS FUNCIONAIS (RF01 a RF07)")
print(f"Razão de Consistência Matemática apurada em {cr_rf:.3f}\n")
display(df_rf)

In [0]:
# ==============================================================================
# BLOCO 5 SIMULAÇÃO DE MONTE CARLO
# ==============================================================================


iteracoes = 10000

def simular_monte_carlo(colunas_bloco, nome_bloco, num_tabela):
    criterios = set()
    pares = []
    
    for col in colunas_bloco:
        match = re.search(r'\[(.*?)\] x \[(.*?)\]', col)
        if match:
            crit_a, crit_b = match.group(1), match.group(2)
            criterios.update([crit_a, crit_b])
            pares.append((col, crit_a, crit_b))
            
    criterios = sorted(list(criterios))
    n = len(criterios)
    resultados_mc = []

    for _ in range(iteracoes):
        matriz_simulada = np.ones((n, n))
        
        for col, crit_a, crit_b in pares:
            valores_numericos = df_ahp[col].apply(converter_escala_saaty).tolist()
            
            valor_sorteado = random.choice(valores_numericos)
            ruido = random.uniform(0.85, 1.15)
            valor_final = valor_sorteado * ruido
            
            i = criterios.index(crit_a)
            j = criterios.index(crit_b)
            matriz_simulada[i, j] = valor_final
            matriz_simulada[j, i] = 1.0 / valor_final
            
        soma_col = matriz_simulada.sum(axis=0)
        mat_norm = matriz_simulada / soma_col
        pesos_simulados = mat_norm.mean(axis=1)
        resultados_mc.append(pesos_simulados)

    df_mc = pd.DataFrame(resultados_mc, columns=criterios)
    
    df_mc_resumo = pd.DataFrame({
        "ID_Requisito": criterios,
        "Peso Médio": df_mc.mean(),
        "Limite Inferior (2,5%)": df_mc.quantile(0.025),
        "Limite Superior (97,5%)": df_mc.quantile(0.975),
        "Desvio Padrão": df_mc.std()
    })
    
    df_mc_resumo = df_mc_resumo.sort_values(by="Peso Médio", ascending=False).reset_index(drop=True)
    df_mc_resumo[['ID', 'Requisito']] = df_mc_resumo['ID_Requisito'].str.split(' - ', n=1, expand=True)
    
    for col in ["Peso Médio", "Limite Inferior (2,5%)", "Limite Superior (97,5%)", "Desvio Padrão"]:
        df_mc_resumo[col] = df_mc_resumo[col].apply(lambda x: f"{x:.2%}")
        
    df_final = df_mc_resumo[['ID', 'Requisito', 'Peso Médio', 'Limite Inferior (2,5%)', 'Limite Superior (97,5%)', 'Desvio Padrão']]
    
    print(f"TABELA {num_tabela} RESULTADO DA SIMULAÇÃO DE MONTE CARLO PARA {nome_bloco.upper()}")
    print(f"({iteracoes} iterações estocásticas computadas)\n")
    display(df_final)
    print("\n" + "="*80 + "\n")

simular_monte_carlo(colunas_rnf, "5 Requisitos Não Funcionais (RNF01 a RNF05)", "11")
simular_monte_carlo(colunas_rf, "7 Requisitos Funcionais (RF01 a RF07)", "12")